# A1.14 · Repudiation and untraceability

**Function A — Securing AI Architectures → CyberTravels' Architecture, and Every Risk It Carries**  ·  *Security of AI*

Builds on **[A1.13 · Resource overload](https://spbreed.github.io/cyber-commons/lessons/A1.13.html)**.

| | |
|---|---|
| Tools used | OpenTelemetry |

## What this lesson is

**What it covers.** Reconstruct who caused a deletion from a log that records only tool calls.

**Why a security engineer needs it.** You cannot say which user caused an action, or what made the agent decide — so the incident cannot be scoped and the action cannot be attributed. The control it builds is: attribution carried on every hop, in a store the agent cannot write to (A2.7).

This is a **risk** lesson: it shows the failure happening before anything tries to stop it, so the control that follows is answering something you have already watched go wrong.

## 1 · The hook

The trace shows the tool call. It does not show the text that motivated the call, or the human the agent was acting for. Six weeks later, nobody can say whether that action was authorised — including the person who authorised it.

> **At CyberTravels.** The log says `cybertravels-svc issued refund 8812`. It does not say which of six people asked, or what text made the agent decide. Six weeks later nobody can tell whether that refund was authorised. R11.

## 2 · The framework

```
   what the trace records          what the question needs
   +----------------------+        +---------------------------+
   | tool: delete_branch  |        | which human asked?         |
   | at: 03:14:22         |        | which agent acted?         |
   | result: ok           |        | under what authority?      |
   +----------------------+        | what input motivated it?   |
                                   +---------------------------+

   complete logs, unanswerable question
```

**OWASP T8 — Repudiation & Untraceability.**

The **observability** component decides whether anything that just happened can
be explained. Most agent logging records tool calls: which tool, what arguments,
what came back. That is enough to debug the agent and not enough to investigate
it.

Three fields are usually missing, and each one removes a different question from
the set you can answer.

**The human principal.** Without it, "which user caused this?" has no answer —
the log says `agent-svc`, as in A1.6.

**The motivating input.** The tool call is recorded; the thing that made the
agent decide to call it is not. So root cause cannot be established at all. You
can see that the agent emailed a file, and nothing tells you the retrieved
document that told it to.

**The delegation chain.** In a multi-agent topology, which hop originated this?
Without the chain you have a set of actions and no order.

There is a fourth problem that is structural rather than about fields: **if the
agent can write to the log store, the log is not evidence.** An agent with
credentials broad enough to be interesting usually has credentials broad enough
to touch the observability stack, and nobody notices until they need the record
to be trustworthy.

> **Where this lands on the reference architecture.**
>
> ```
> ingress -> orchestrator -> agent_runtime -> model
>                                |              |
>                          messaging        tools / mcp
>                                |              |
>                       knowledge / memory   egress
>            identity + policy wrap every call · observability records it
> ```

## 3 · The risk, realised

A deletion happened. Answer three questions from the log you have.

## 4 · The check, as a skill

CyberTravels logs every tool call. The skill puts the three questions an investigation opens with to one real record, and reports per question the field that would have answered it.

### The skill — [`skills/threats/audit-answerability-check/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/threats/audit-answerability-check/SKILL.md)

```yaml
name: audit-answerability-check
description: >-
  Put an investigation's three questions to an existing agent log — which human,
  what motivated the action, which hop originated it — and record which of them
  the record cannot answer. Use when reviewing agent telemetry before an
  incident rather than during one.
allowed-tools: Read, Grep, Glob
```

# A complete log that answers nothing

Agent logs are usually complete in the sense that every tool call is present.
They are useless in the sense that the three questions an investigation opens
with have no fields behind them. The check is not "are we logging" — it is
"which question dies here".

## When to use this

Before an incident. Run it against a real log line from production, not against
the logging design.

## Procedure

**1 — Take one real record.** A single tool-call row, with every field it
actually carries. Not the schema — the row.

**2 — Ask: which human?** Is there a field naming the principal on whose behalf
this ran? An agent identity is not an answer; it is the thing that ran.

**3 — Ask: what motivated it?** Which input caused this call — the user's
request, a retrieved document, a tool result, a memory record. Without it you
cannot tell an instructed action from an injected one, and that is the
distinction the whole investigation turns on.

**4 — Ask: which hop originated it?** In any multi-agent or delegated flow,
which agent started the chain. A chain reconstructed by correlating timestamps
across four services is a chain you will not reconstruct at 2am.

**5 — Report per question, with the field that would answer it.** Three rows.
Each says: answerable yes/no, and the field to add. Anything vaguer produces a
logging project rather than a fix.

## Example

**Input** — the fixture committed at the top of [`scripts/audit_answerability_check.py`](scripts/audit_answerability_check.py). Edit it and re-run: the buckets, counts and verdicts below are derived from it, not hard-coded.

**Output** — the opening lines of a real run:

```
the log you have:
   09:14:02  agent-svc search      {'q': 'invoice 8812'}
   09:14:07  agent-svc fetch_doc   {'id': 'wiki/473'}
   09:14:11  agent-svc run_query   {'sql': 'DELETE FROM invoices WHERE id=8812'}
   09:14:12  agent-svc send_email  {'to': 'ops@corp.example'}

question                                    field needed        present?
which user caused the deletion?             principal           NO
```

The run continues past this. The script is the example: `test_skills.py` executes it on every build, so this block cannot drift from what the skill actually prints.

## Output contract

```json
{
  "record": {"fields": ["str"]},
  "questions": [{"question": "which human|what motivated|which hop", "answerable": false, "missing_field": "str"}],
  "answerable_count": 0,
  "reconstruction": {"possible": false, "requires_correlating": ["str"]}
}
```

## Failure modes

- **Checking the schema.** Fields exist in schemas and are null in rows.
- **Accepting the agent identity as the principal.** It answers "what", never
  "who".
- **Recording "add more logging".** Name the three fields or nothing changes.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/threats/audit-answerability-check/scripts/audit_answerability_check.py
SCRIPT = "skills/threats/audit-answerability-check/scripts/audit_answerability_check.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; `sparse-checkout set skills` then materialises only what runs.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set", "skills"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

A complete-looking tool-call log answers none of the three questions an investigation needs — which user, what motivated it, which hop originated it — because the principal, the motivating input and the delegation chain were never recorded.

## Your turn

Take yesterday's agent logs and try to answer 'which user caused this action'. Time how long it takes. That number is your time-to-attribution during an incident, when it will be worse.

---

**Next → [A1.15 · Overwhelming the human in the loop](https://spbreed.github.io/cyber-commons/lessons/A1.15.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A1.14.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A1.14.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*